# GeoVision — Train MobileNetV2 (13-channel) on EuroSAT MS

Runs on Google Colab T4 GPU. ~20–30 min for 20 epochs (MobileNetV2 is lighter than ResNet50).

Same flow as the PyTorch notebook — swap `--framework pytorch` for `--framework tensorflow`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/GeoVision'
os.makedirs(f'{DRIVE_ROOT}/data/raw', exist_ok=True)

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/yourname/geovision.git'
REPO_DIR = f'{DRIVE_ROOT}/code'

if REPO_URL and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif REPO_URL:
    !git -C {REPO_DIR} pull

assert os.path.exists(REPO_DIR), 'Set REPO_URL above, or upload code/ to Drive first'
%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
%pip install -q 'rasterio>=1.4.0' 'albumentations>=1.4.0' 'mlflow>=2.18.0'

In [ ]:
import glob

EUROSAT_URL = 'https://madm.dfki.de/files/sentinel/EuroSATallBands.zip'
ZIP_PATH = f'{DRIVE_ROOT}/data/raw/EuroSATallBands.zip'
EXTRACT_DIR = f'{DRIVE_ROOT}/data/raw'

candidates = glob.glob(f'{EXTRACT_DIR}/**/Forest', recursive=True)
if not candidates:
    if not os.path.exists(ZIP_PATH):
        print('Downloading EuroSAT MS (~3 GB)...')
        !wget -q -O {ZIP_PATH} {EUROSAT_URL}
    print('Extracting...')
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
    candidates = glob.glob(f'{EXTRACT_DIR}/**/Forest', recursive=True)

DATA_ROOT = os.path.dirname(candidates[0])
print('Data root:', DATA_ROOT)
!ls {DATA_ROOT}

In [ ]:
!python -m src.train --framework tensorflow --data-root {DATA_ROOT} --epochs 20 --batch-size 64 --lr 1e-4

In [ ]:
!python -m src.evaluate --framework tensorflow --data-root {DATA_ROOT} --ckpt models/mobilenetv2_13ch.keras

## Done

- **Weights:** `code/models/mobilenetv2_13ch.keras` (~30 MB)
- **Metrics:** `code/data/metrics/confusion_matrix_tensorflow.{csv,png}`, `summary_tensorflow.csv`, `classification_report_tensorflow.csv`
- **MLflow runs:** `code/mlruns/`